# Step 8: Multi-Attribute / Priority Index – Narrative Summary

## Purpose

The goal of Step 8 is to generate a **single, composite priority index** for each dwelling in the Fintry EPC dataset. This index combines multiple dimensions—energy efficiency improvement potential, environmental impact reduction, and financial cost-effectiveness—into a single numeric score that can guide **retrofit prioritization**. 

By producing this index, we can identify homes that offer the **highest overall potential for improvement**, balancing energy, carbon, and cost considerations.

---

## Logic Behind the Index

The **priority index** is calculated in several steps:

1. **Selection of Key Metrics**

   We focus on numeric indicators that capture the most relevant aspects of retrofit prioritization:

   - `DELTA_ENERGY_EFFICIENCY`: Expected improvement in energy efficiency from potential measures.  
   - `DELTA_ENVIRONMENT_IMPACT`: Expected reduction in environmental impact (CO₂) from potential measures.  
   - `NPV`: Net present value of retrofit measures, representing financial feasibility.  
   - `COST_PER_KWH_REDUCED`: Cost-effectiveness in terms of energy saved per pound spent.  
   - `COST_PER_TON_CO2_REDUCED`: Cost-effectiveness in terms of carbon saved per pound spent.  

   Missing values are filled with zero to allow all dwellings to be included in the analysis.

2. **Scaling / Normalization**

   These metrics have **different units and ranges**, so we scale each column to the range 0–1 using Min-Max normalization:

   \[
   x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
   \]

   This ensures that each metric contributes proportionally and comparably to the final index.

3. **Weighted Combination**

   Each scaled metric is assigned a **weight** reflecting its relative importance in prioritization. For example:

   | Metric                         | Weight |
   |--------------------------------|--------|
   | DELTA_ENERGY_EFFICIENCY         | 0.3    |
   | DELTA_ENVIRONMENT_IMPACT        | 0.3    |
   | NPV                             | 0.1    |
   | COST_PER_KWH_REDUCED             | 0.15   |
   | COST_PER_TON_CO2_REDUCED         | 0.15   |

   The **priority index** for each dwelling is then calculated as:

   \[
   \text{PRIORITY_INDEX} = 
   (0.3 \times \text{DELTA_ENERGY_EFFICIENCY}_\text{scaled}) +
   (0.3 \times \text{DELTA_ENVIRONMENT_IMPACT}_\text{scaled}) +
   (0.1 \times \text{NPV}_\text{scaled}) +
   (0.15 \times \text{COST_PER_KWH_REDUCED}_\text{scaled}) +
   (0.15 \times \text{COST_PER_TON_CO2_REDUCED}_\text{scaled})
   \]

   The index is therefore a **weighted aggregate of all key performance metrics**, normalized to give dwellings with the best combined energy, environmental, and financial performance the highest score.

4. **Interpretation**

   - The **final index value** ranges from 0 to 1.  
   - **Higher values** indicate **higher overall retrofit priority**, meaning the dwelling has a strong combination of:  
     - High energy efficiency improvement potential  
     - Significant CO₂ reduction potential  
     - Financial feasibility or cost-effectiveness  

   - **Top 10% of dwellings** by index are identified as **high-priority targets** for interventions.  

5. **Outcome**

   This approach allows us to:

   - Combine multiple objectives into a **single, actionable metric**.  
   - Compare dwellings objectively across the entire community.  
   - Provide clear guidance for **community-level decision-making** and for visualization in dashboards or reports.  

---

In summary, the **priority index synthesizes energy, carbon, and cost metrics into one score per dwelling**, enabling efficient and data-driven prioritization of retrofit measures for the Fintry community.

In [1]:
# Step 8a: Multi-Attribute / Priority Indices
# Description: Import libraries and load the processed EPC dataset for analysis.

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load the dataset
file_path = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_step4_building_characteristics.csv"
df = pd.read_csv(file_path)

# Quick check of dataset
print("Dataset shape:", df.shape)
print(df.columns[:20])  # preview first 20 columns

Dataset shape: (117, 157)
Index(['BUILDING_REFERENCE_NUMBER', 'OSG_REFERENCE_NUMBER', 'ADDRESS1',
       'ADDRESS2', 'ADDRESS3', 'POSTCODE', 'LATITUDE', 'LONGITUDE',
       'INSPECTION_DATE', 'TYPE_OF_ASSESSMENT', 'LODGEMENT_DATE',
       'ENERGY_CONSUMPTION_CURRENT', 'TOTAL_FLOOR_AREA',
       '3_YR_ENERGY_COST_CURRENT', '3_YR_ENERGY_SAVINGS_POTENTIAL',
       'CURRENT_ENERGY_EFFICIENCY', 'CURRENT_ENERGY_RATING',
       'POTENTIAL_ENERGY_EFFICIENCY', 'POTENTIAL_ENERGY_RATING',
       'ENVIRONMENT_IMPACT_CURRENT'],
      dtype='object')


In [2]:
# Step 8b: Define columns to use for priority index
# Description: Choose numeric columns reflecting improvement potential, cost-effectiveness, and CO2 reduction.

# Example columns: adjust if needed based on your dataset
priority_columns = [
    "DELTA_ENERGY_EFFICIENCY",       # improvement potential (energy efficiency delta)
    "DELTA_ENVIRONMENT_IMPACT",      # environmental impact reduction
    "NPV",                           # financial metric (negative NPV indicates less favorable)
    "COST_PER_KWH_REDUCED",          # cost-effectiveness per kWh
    "COST_PER_TON_CO2_REDUCED"       # cost-effectiveness per ton CO2
]

# Handle missing values
df[priority_columns] = df[priority_columns].fillna(0)

In [3]:
# Step 8c: Scale / normalize numeric columns
# Description: Scale each column 0-1 for combination into a single weighted priority index.

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(df[priority_columns])

# Create DataFrame with scaled values
df_scaled = pd.DataFrame(scaled_values, columns=[col + "_scaled" for col in priority_columns])
df = pd.concat([df, df_scaled], axis=1)

In [4]:
# Step 8d: Define weighted priority index
# Description: Combine scaled columns into a single index. Weights can be adjusted as desired.

# Example weights: sum must be 1. Adjust according to FDT priorities
weights = {
    "DELTA_ENERGY_EFFICIENCY_scaled": 0.3,
    "DELTA_ENVIRONMENT_IMPACT_scaled": 0.3,
    "NPV_scaled": 0.1,
    "COST_PER_KWH_REDUCED_scaled": 0.15,
    "COST_PER_TON_CO2_REDUCED_scaled": 0.15
}

# Compute priority index
df["PRIORITY_INDEX"] = (
    df["DELTA_ENERGY_EFFICIENCY_scaled"] * weights["DELTA_ENERGY_EFFICIENCY_scaled"] +
    df["DELTA_ENVIRONMENT_IMPACT_scaled"] * weights["DELTA_ENVIRONMENT_IMPACT_scaled"] +
    df["NPV_scaled"] * weights["NPV_scaled"] +
    df["COST_PER_KWH_REDUCED_scaled"] * weights["COST_PER_KWH_REDUCED_scaled"] +
    df["COST_PER_TON_CO2_REDUCED_scaled"] * weights["COST_PER_TON_CO2_REDUCED_scaled"]
)

In [5]:
# Step 8e: Generate descriptive statistics for the priority index
# Description: Examine overall distribution and identify top-priority dwellings.

print("Priority Index Descriptive Stats:")
print(df["PRIORITY_INDEX"].describe())

# Identify top 10% priority dwellings
top_threshold = df["PRIORITY_INDEX"].quantile(0.9)
top_dwellings = df[df["PRIORITY_INDEX"] >= top_threshold]

print(f"Number of top 10% dwellings: {top_dwellings.shape[0]}")
top_dwellings[["BUILDING_REFERENCE_NUMBER", "ADDRESS1", "PRIORITY_INDEX"]].head(10)

Priority Index Descriptive Stats:
count    117.000000
mean       0.359250
std        0.133765
min        0.145041
25%        0.276243
50%        0.346080
75%        0.420395
max        0.768494
Name: PRIORITY_INDEX, dtype: float64
Number of top 10% dwellings: 12


,BUILDING_REFERENCE_NUMBER,ADDRESS1,PRIORITY_INDEX
20,1001870854,THE BYRE 3 KILUNAN STEADING,0.615078
22,1002143293,15 MENZIES AVENUE,0.561382
24,1002143353,5 MENZIES AVENUE,0.614180
30,1001829085,9 CULCREUCH AVENUE,0.681727
47,1001951889,1 CROFTINSTILLY,0.680020
66,1001951902,GREENVALE,0.583189
93,1001951906,JAW FARM COTTAGE,0.669112
95,1000238914,53 MAIN STREET,0.768494
104,1001664934,23 DUNMORE GARDENS,0.592860
106,1001951903,HALLBURN,0.534610


In [6]:
# Step 8f: Optional - save results to CSV
# Description: Export the dataset including the new PRIORITY_INDEX for use in visualization or web dashboards.

output_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_priority_index_step8.csv"
df.to_csv(output_file, index=False)
print("Saved priority index dataset to:", output_file)

Saved priority index dataset to: /Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_priority_index_step8.csv
